## Pre-processing Notebook

This notebook contains the instructions on how to process the data once it is downloaded, download instructions can be found [here](https://github.com/ppuentex/detection_comparison-CRB/blob/main/code/download_instructions.ipynb). In this notebook, you will find the following: 

1. Instruction on cropping the data for a given watershed boundary (ie. shapefile). 
2. Calculate the mode (most frequent pixels) for a given time period using Dask and Numba. 
3. Reprojecting raster data to have the same projection in meters, resolution of pixels is either 10m or 30m, and make sure nothing is shifted in the process.
4. Calculating the Kronecker product to get downscale Landsat 30m pixels to match the Sentinel 10m pixels, without having to reclassify. 
5. Change the classifications of Sentinel and Landsat to binary, inundated or not inundated. 
6. Create a rasterized version of the HUC4 shapefile geometries for analysis.

*Note: These instructions are written for the Colorado River Basin specifically but can be applied elsewhere with a new changes on area of interest when downloading.*

These instructions assume you have already combined the tiles through mosaicking them, it is recommended to use QGIS or use virtual merge through `gdalbuildvrt` [documentation here](https://gdal.org/en/stable/programs/gdalbuildvrt.html). 

Below is an example snippet to do it using Python. 

```
from pathlib import Path
import subprocess

#List of cropped tif files
folder_path = Path('./data')
vrt_path = folder_path / "merged.vrt"

tif_files = [str(tif) for tif in folder_path.glob('*_cropped_10m.tif')]
subprocess.run(["gdalbuildvrt", str(vrt_path)] + [str(file) for file in tif_files])

```
The code above creates the virtual .tif file and runs smoothly on your local machine. Next, this can be saved to your local machine in a compressed form, this does not lose any information. The code below saves your mosaic file that is necessary. 

```
import rasterio as rio
from rasterio.enums import Resampling

output_path = "./data/merged_land_cover_2019.tif"

with rio.open(vrt_path) as src:
    profile = src.profile.copy()
    profile.update({
        "driver": "GTiff",
        "compress": "LZW",
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512
    })

    with rio.open(output_path, "w", **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window=window, resampling=Resampling.nearest)
            dst.write(data, window=window, indexes=1)
```

### 1. Cropping the data for a given watershed boundary 
This technique was used to create all the raster files in this study. The cropped versions of the landsat yearly datasets for 2015 to 2021 can be found in `data/landsat-yearly/` and the cropped version of the sentinel data can be found in `data/zenodo-data/crb_sentinel_water-extent.tif`. 

In [ ]:
import subprocess

subprocess.run([
    "gdalwarp",
    "-cutline", "./data/shapefiles/crb_boundary.shp",
    "-crop_to_cutline",
    "-dstnodata", "0",
    "-co", "COMPRESS=LZW",
    "-co", "TILED=YES",
    "<input tif file path>", #input file 
    "<output tif file path>" #output file
])

### 2. Calculating the Mode using Dask and Numba

The Landsat file of the mode calculation from 2015 to 2021 can be found in `data/zenodo-data/mode_crb_landsat.tif`. Note that this is not the version that is used in the analysis due to still needing to reproject and resample to meet the Sentinel resolution. 

In [ ]:
from pathlib import Path 
import rioxarray
import numpy as np
import dask.array as da 
import numba 
from dask.diagnostics import ProgressBar
import rasterio

mode_output_file = '../data/zenodo-data/mode_landsat.tif'

folder_path = Path('../data/landsat-yearly')

tif_files = [str(tif_file) for tif_file in folder_path.glob('*.tif')]

tif_files #list of files that we will calculate the mode for 

['../data/landsat-yearly/2019_CRB.tif',
 '../data/landsat-yearly/2018_CRB.tif',
 '../data/landsat-yearly/2015_CRB.tif',
 '../data/landsat-yearly/2017_CRB.tif',
 '../data/landsat-yearly/2021_CRB.tif',
 '../data/landsat-yearly/2020_CRB.tif',
 '../data/landsat-yearly/2016_CRB.tif']

In [35]:
#stack the files using dask 
data_arrays = [rioxarray.open_rasterio(fp, masked = True).squeeze().chunk({"x": 1000, "y": 1000}) for fp in tif_files]
stacked_data = da.stack(data_arrays,axis=-1) #convert to dask array
stacked_data = stacked_data.astype(np.uint8)
stacked_data.shape

(53809, 40320, 7)

In [ ]:
#Using Numba to improve the efficiency of the mode calculation 
@numba.jit(nopython=True, parallel=True)
def numba_mode(arr):
    """Computes mode along the last axis using Numba."""
    h, w = arr.shape  # height, width 
    mode_result = np.zeros((h, w), dtype=np.uint8)
    
    for i in numba.prange(h):  # Iterate over rows
        for j in numba.prange(w):  # Iterate over cols
            values = arr[i,j,:].astype(np.uint8)
            # Count the occurrences using np.bincount
            counts = np.bincount(values)
            
            #find the mode (value with the max count)
            mode_result[i, j] = np.argmax(counts)  
    return mode_result


#Compute Mode Efficiently (should be much faster)
with ProgressBar():
    stacked_numpy = stacked_data.compute()
    mode_result = numba_mode(stacked_numpy)

In [ ]:
#save the array with the original metadata from the .tiff file 
with rasterio.open(tif_files[0]) as src:
    profile = src.meta.copy()
        
    #update metadata 
    profile.update({'count': 1}) #specify the number of bands
    profile.update({"compress": "lzw"}) 

    #write 
    with rasterio.open(mode_output_file, 'w', **profile) as dst:
        dst.write(mode_result, 1) #write data into first band

### 3. Reprojecting Rasters so that both are in meters and resolution is 10m and 30m for Sentinel and Landsat respectively

Use the following code to make sure the CRS for both rasters are EPSG: 3857.

Note that at this point all files have the original classifications where Landsat pixels are 1:no water, 2:seasonal water, 3:permanent water, 4:no observation or Sentinel pixels are 1:permanent water, 2:seasonal water, 3:land. 

The code below reprojects the raster files to meters and makes sure the resolution is 10m for Sentinel and 30m for Landsat. 

In [ ]:
import rasterio 
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np 

landsat_input = '../data/zenodo-data/mode_crb_landsat.tif'
landsat_output = '../data/zenodo-data/mode_crb_landsat_3857_30m.tif'

desired_res = 30

#open raster file 
with rasterio.open(landsat_input) as src:
    #destination crs 
    dst_crs = 'EPSG:3857'

    #calcualte the tranform for the new CRS
    transform, width, height = calculate_default_transform(src.crs,dst_crs,src.width,src.height, *src.bounds, resolution=desired_res)

    #print(width)
    #print(height)

    #set the output raster properties
    kwargs = src.meta.copy()

    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height,
        "compress": "lzw"
    })

    print(src.count)

    #create the output raster file and reproject 

    with rasterio.open(landsat_output, 'w', **kwargs) as dst: 
        reproject(
            source=rasterio.band(src,1),
            destination= rasterio.band(dst,1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest) #resampling nearest is best for categorical, preserves pixel classification

In [ ]:
sentinel_input = '../data/zenodo-data/crb_sentinel_water-extent.tif'
sentinel_output = '../data/zenodo-data/crb_sentinel_water-extent3857_10m.tif'

desired_res = 10

#open raster file 
with rasterio.open(sentinel_input) as src:
    #destination crs 
    dst_crs = 'EPSG:3857'

    #calcualte the tranform for the new CRS
    transform, width, height = calculate_default_transform(src.crs,dst_crs,src.width,src.height, *src.bounds, resolution= desired_res)

    #print(width)
    #print(height)

    #set the output raster properties
    kwargs = src.meta.copy()

    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height,
        "compress": "lzw"
    })

    #print(src.count)

    #create the output raster file and reproject 

    with rasterio.open(sentinel_output, 'w', **kwargs) as dst: 
        reproject(
            source=rasterio.band(src,1),
            destination= rasterio.band(dst,1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest)#resampling nearest is best for categorical, preserves pixel classification

### 4. Calculate the Kronecker product to turn Landsat from 30 meters to 10 meters. 
This is done by expanding each 30 meter pixel into a 3x3 grid of 10 m pixels where each original value is repeated. Using `np.kron` duplicates the exaxt pixel and is fast and memory-efficient. 

In [ ]:
landsat_30m_file = '../data/zenodo-data/mode_crb_landsat_3857_30m.tif'

#open tiff file and read it as an array 
with rasterio.open(landsat_30m_file) as dataset: 
    raster_array = dataset.read(1)

# Print shape and check the array
print(raster_array.shape)  # (height, width)

In [ ]:
upscale_factor = 3

height, width = raster_array.shape

new_height, new_width = height*upscale_factor, width*upscale_factor

#create a copy of the metadata in the original landsat@30m 
new_meta = dataset.meta.copy()

#create an empty array for the upsampled image 
upsampled_array = np.zeros((new_height, new_width), dtype = raster_array.dtype)


print(new_height, new_width) #this is what the new height and width of the array will be 

In [ ]:
#processing the kronecker product in chunks

#define the chunk size for processing
chunk_size = 1024

for y in range(0, height, chunk_size):
    for x in range(0, width, chunk_size):

        #define chunk boundaries 
        y_end = min(y+chunk_size, height)
        x_end = min(x+chunk_size, width)

        #Exact chunk 
        chunk = raster_array[y:y_end, x:x_end]

        #apply the 3x3 upsampling 
        upsampled_chunk = np.kron(chunk, np.ones((upscale_factor, upscale_factor), dtype=chunk.dtype))

        #place upsampled chunk into correct location in the final array 
        upsampled_array[
            y * upscale_factor : y_end * upscale_factor,
            x * upscale_factor : x_end * upscale_factor
        ] = upsampled_chunk

In [ ]:
from rasterio.transform import Affine
#update metadata to save landsat@10m 
new_meta.update({
    "height": upsampled_array.shape[0],
    "width": upsampled_array.shape[1],
    "transform": Affine(
        new_meta["transform"].a / upscale_factor,  # Scale pixel size from 30m → 10m
        new_meta["transform"].b,
        new_meta["transform"].c,
        new_meta["transform"].d,
        new_meta["transform"].e / upscale_factor,  # Scale pixel size from 30m → 10m
        new_meta["transform"].f
    ),
    "dtype": str(upsampled_array.dtype),  # Ensure correct dtype
    "compress": "lzw" #can't forget the compression
})

In [ ]:
#save the upsampled_array as a raster 
output_path = '../data/zenodo-data/mode_crb_landsat_3857_10m.tif'

with rasterio.open(output_path, 'w', **new_meta) as dst:
    dst.write(upsampled_array, 1)

print("Upsampled Raster saved with updated meta")

### 5. Binary inundation

For this study we are only interested in whether the pixel is inundated or not. Therefore, the create new raster files that will show inundation or not inundation for Sentinel and Landsat. 

Landsat classification </br>
2:seasonal water & 3:permanent water &rarr;  1: inundated </br>
1: no water & 4: no observation &rarr; 0: not inundated

Sentinel classification </br>
2:seasonal water & 1:permanent water &rarr;  1: inundated </br>
3: land &rarr; 0: not inundated

The final versions of these files can be found in `data/zenodo-data/` as `sentinel_extent_inundation.tif` and `landsat_mode_inundation.tif`. Note that for graphing purposes these .tif files are large and hard to see on maps, therefore there exists a shapefile version of these files in `data/zenodo-data/shapefiles`. 

In [ ]:
#create function to reclassify 
import rioxarray as rxr
import xarray as xr

def reclassify_inundation(input_path, output_path, inundated_classes):
        """
        Reclassifies inundation raster to binary (1 = inundated, 0 = not).
        Preserves width, height, CRS, transform, dtype, and metadata.

        Parameters:
        - input_path: str, path to input raster (.tif)
        - output_path: str, path to output reclassified raster
        - inundated_classes: list of int, pixel values considered inundated
        """

        # Open raster lazily with Dask
        da = rxr.open_rasterio(input_path, masked=True, chunks='auto').squeeze()

        # Binary reclassification
        binary = xr.where(da.isin(inundated_classes), 1, 0).astype('uint8')

        # Set attributes for output
        binary.rio.set_nodata(0, inplace=True)
        binary.rio.write_crs(da.rio.crs, inplace=True)

        # Write to GeoTIFF with LZW compression
        binary.rio.to_raster(output_path, compress='LZW')

        # print(f"Saved binary raster to: {output_path}")

In [ ]:
#for landsat 
reclassify_inundation('../data/zenodo-data/mode_crb_landsat_3857_10m.tif', '../data/zenodo-data/landsat_mode_inundation.tif', [2,3])

#for sentinel
reclassify_inundation('../data/zenodo-data/crb_sentinel_water-extent3857_10m.tif', '../data/zenodo-data/sentinel_extent_inundation.tif', [1,2])

### 6. Create a raster for HUC4 geometries

In order to do the analysis of overlap between the Sentinel and Landsat, it is necessary to turn the HUC4 geometries into a raster so that you are working with rasters only and not having to project the rasters to make sure they overlap with the shapefile. This can make you run into problems with the kernel, therefore we found that the best and easiest way around this is to create a rasterized version of the HUC4 shapefile. 

In [37]:
from rasterio.features import rasterize
import geopandas as gpd 
from shapely.geometry import mapping

#this cell is necessary to not get any errors when loading shapefiles 
import os
os.environ["PROJ_DATA"] = "/Users/ppuente/github/detection_comparison-CRB/comparison-env/lib/python3.11/site-packages/fiona/proj_data"
# replace the /Users/ppuente/github/detection_comparison-CRB/comparison-env to your own path and what you named the your environment (ie. comparison-env)

In [ ]:
#load the shapefile with hucs
huc4_path = '../data/shapefiles/crb_huc4.shp'
crb_huc4 = gpd.read_file(huc4_path)

output_raster = '../data/zenodo-data/huc4_raster.tif'

# load the raster file so we know what we are comparing it to
with rasterio.open('../data/zenodo-data/sentinel_extent_inundation.tif') as src:
    raster = src.read(1)
    transform = src.transform
    out_meta = src.meta.copy()

#first we need to do label mapping, we are replacing the huc4 id's for rasterizing the shapefile to keep within uint8 to save memory
label_mapping = {1401: 1, 1402: 2, 1403: 3, 1404: 4, 1405: 5, 1406: 6, 1407:7, 1408: 8,
                 1501: 11, 1502: 12, 1503: 13, 1504: 14, 1505: 15, 1506:16, 1507:17}

# load shapefile to create a raster to keep track of which huc id the comparisons are occurring
huc_shapes = [(mapping(row.geometry), label_mapping.get(int(row['huc4']))) for _, row in crb_huc4.iterrows()]

huc_raster = rasterize(huc_shapes, raster.shape, fill=0, transform=transform, dtype=np.uint8)

In [ ]:
out_meta.update({
            "height": huc_raster.shape[0],
            "width": huc_raster.shape[1],
            "compress": "lzw",  # Keep LZW compression
            "crs": src.crs,
            "dtype": "uint8",
            "transform": transform
        })

with rasterio.open(output_raster, 'w', **out_meta) as dst:
        dst.write(huc_raster,1)
        print(f'saved for {output_raster}')